# Mondial-Xboost — Entrenamiento Iterativo

**Setup:**
1. Subir `all_matches.parquet` a `/content/drive/MyDrive/Mondial-Xboost/data/`
2. Runtime → T4 GPU
3. Cambiar `BATCH_NUM` en cada ejecución
4. Runtime → Run All

In [ ]:
# CELDA 1: SETUP
!pip install xgboost lightgbm catboost optuna scikit-learn pandas numpy pyarrow -q

import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import json
import time
import os
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss
from collections import defaultdict
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ═══ CONFIG ═══
BATCH_NUM = 1
TRIALS_PER_MODEL = 20
TEST_CUTOFF = '2024-01-01'
VAL_CUTOFF = '2023-01-01'
# ═════════════

print(f'Batch {BATCH_NUM} | {TRIALS_PER_MODEL} trials x 5 = {TRIALS_PER_MODEL*5} total')

In [ ]:
# CELDA 2: CARGAR DATOS
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/Mondial-Xboost/data/all_matches.parquet'
print(f'Leyendo: {DATA_PATH}')

raw = pd.read_parquet(DATA_PATH)
raw['date'] = pd.to_datetime(raw['date'], errors='coerce')
raw = raw.dropna(subset=['date','home_team','away_team','home_goals','away_goals'])
raw = raw.sort_values('date').reset_index(drop=True)
raw['home_goals'] = raw['home_goals'].astype(int)
raw['away_goals'] = raw['away_goals'].astype(int)
raw['outcome'] = np.where(raw['home_goals']>raw['away_goals'], 0,
                 np.where(raw['home_goals']==raw['away_goals'], 1, 2))

DATA_INFO = {
    'total': len(raw),
    'leagues': raw['league_code'].nunique(),
    'from': str(raw['date'].min().date()),
    'to': str(raw['date'].max().date()),
    'home_wins': int((raw.outcome==0).sum()),
    'draws': int((raw.outcome==1).sum()),
    'away_wins': int((raw.outcome==2).sum())
}
print(f'Dataset: {DATA_INFO["total"]} partidos, {DATA_INFO["leagues"]} ligas')
print(f'Rango: {DATA_INFO["from"]} -> {DATA_INFO["to"]}')

In [ ]:
# CELDA 3: FEATURE ENGINEERING
def compute_elo(df):
    ratings = defaultdict(lambda: 1500.0)
    he, ae = [], []
    for row in df.itertuples(index=False):
        h, a = row.home_team, row.away_team
        rh, ra = ratings[h], ratings[a]
        he.append(rh)
        ae.append(ra)
        K = 60 if 'WC' in str(getattr(row, 'league_code', '')) else 30
        dr = (rh - ra) + 100
        eh = 1.0 / (1.0 + 10.0 ** (-dr / 400.0))
        sh = 1.0 if row.home_goals > row.away_goals else (0.5 if row.home_goals == row.away_goals else 0.0)
        ratings[h] = rh + K * (sh - eh)
        ratings[a] = ra - K * (sh - eh)
    df = df.copy()
    df['home_elo'] = he
    df['away_elo'] = ae
    df['elo_diff'] = df['home_elo'] - df['away_elo']
    return df

def compute_rolling(df):
    home = df[['date','home_team','away_team','home_goals','away_goals']].copy()
    home['team'] = home['home_team']
    home['gs'] = home['home_goals']
    home['gc'] = home['away_goals']
    home['pts'] = np.where(home['home_goals']>home['away_goals'], 3,
                   np.where(home['home_goals']==home['away_goals'], 1, 0))
    away = df[['date','home_team','away_team','home_goals','away_goals']].copy()
    away['team'] = away['away_team']
    away['gs'] = away['away_goals']
    away['gc'] = away['home_goals']
    away['pts'] = np.where(away['away_goals']>away['home_goals'], 3,
                   np.where(away['home_goals']==away['away_goals'], 1, 0))
    long = pd.concat([home, away]).sort_values(['team', 'date'])
    for w in [5, 10]:
        long[f'pa{w}'] = long.groupby('team')['pts'].transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean() / 3.0)
        long[f'gsa{w}'] = long.groupby('team')['gs'].transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        long[f'gca{w}'] = long.groupby('team')['gc'].transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        long[f'wr{w}'] = long.groupby('team')['pts'].transform(lambda s: s.shift(1).rolling(w, min_periods=1).apply(lambda x: (x==3).sum()) / s.shift(1).rolling(w, min_periods=1).count())
        long[f'dr{w}'] = long.groupby('team')['pts'].transform(lambda s: s.shift(1).rolling(w, min_periods=1).apply(lambda x: (x==1).sum()) / s.shift(1).rolling(w, min_periods=1).count())
        long[f'lr{w}'] = long.groupby('team')['pts'].transform(lambda s: s.shift(1).rolling(w, min_periods=1).apply(lambda x: (x==0).sum()) / s.shift(1).rolling(w, min_periods=1).count())
    long['mp'] = long.groupby('team').cumcount()
    return long

def compute_h2h(df):
    df = df.copy()
    df['pk'] = df.apply(lambda r: tuple(sorted([r['home_team'], r['away_team']])), axis=1)
    df = df.sort_values(['pk', 'date']).reset_index(drop=True)
    prev = {}
    h2h_l, h2h_g, h2h_w, h2h_y = [], [], [], []
    for _, row in df.iterrows():
        k = row['pk']
        if k in prev:
            p = prev[k]
            h2h_l.append(p['rv'])
            h2h_g.append(p['ag'])
            h2h_w.append(p['wd'])
            h2h_y.append((row['date'] - p['dt']).days / 365.25)
        else:
            h2h_l.append(np.nan)
            h2h_g.append(np.nan)
            h2h_w.append(np.nan)
            h2h_y.append(np.nan)
        rv = 1.0 if row['home_goals'] > row['away_goals'] else (0.5 if row['home_goals'] == row['away_goals'] else 0.0)
        prev[k] = {'rv': rv, 'ag': row['home_goals'] + row['away_goals'], 'wd': 1 if rv == 1 else (-1 if rv == 0 else 0), 'dt': row['date']}
    df['h2h_last'] = h2h_l
    df['h2h_goals'] = h2h_g
    df['h2h_wins'] = h2h_w
    df['h2h_years'] = h2h_y
    return df

t0 = time.time()
df = compute_elo(raw)
print(f'Elo: {time.time()-t0:.0f}s')
t0 = time.time()
long = compute_rolling(df)
print(f'Rolling: {time.time()-t0:.0f}s')
t0 = time.time()
df = compute_h2h(df)
print(f'H2H: {time.time()-t0:.0f}s')

hc = ['date','team','pa5','pa10','gsa10','gca10','wr10','dr10','lr10','mp']
home_merge = long[hc].rename(columns={
    'team': 'home_team', 'pa5': 'h_pa5', 'pa10': 'h_pa10',
    'gsa10': 'h_gsa10', 'gca10': 'h_gca10', 'wr10': 'h_wr10',
    'dr10': 'h_dr10', 'lr10': 'h_lr10', 'mp': 'h_mp'
})
away_merge = long[hc].rename(columns={
    'team': 'away_team', 'pa5': 'a_pa5', 'pa10': 'a_pa10',
    'gsa10': 'a_gsa10', 'gca10': 'a_gca10', 'wr10': 'a_wr10',
    'dr10': 'a_dr10', 'lr10': 'a_lr10', 'mp': 'a_mp'
})
df = df.merge(home_merge, on=['date', 'home_team'], how='left')
df = df.merge(away_merge, on=['date', 'away_team'], how='left')

FC = ['elo_diff','home_elo','away_elo','h_pa5','h_pa10','h_gsa10','h_gca10',
      'h_wr10','h_dr10','h_lr10','h_mp','a_pa5','a_pa10','a_gsa10','a_gca10',
      'a_wr10','a_dr10','a_lr10','a_mp','h2h_last','h2h_goals','h2h_wins','h2h_years']
avail = [c for c in FC if c in df.columns]
df = df.dropna(subset=avail + ['outcome'])
FEAT_INFO = {'features': len(avail), 'rows': len(df)}
print(f'Features: {len(avail)} cols, {len(df)} rows')

In [ ]:
# CELDA 4: SPLIT TEMPORAL 3-WAY
train = df[df['date'] < VAL_CUTOFF]
val = df[(df['date'] >= VAL_CUTOFF) & (df['date'] < TEST_CUTOFF)]
test = df[df['date'] >= TEST_CUTOFF]

X_train, y_train = train[avail].fillna(0), train['outcome'].astype(int)
X_val, y_val = val[avail].fillna(0), val['outcome'].astype(int)
X_test, y_test = test[avail].fillna(0), test['outcome'].astype(int)

SPLIT_INFO = {'train': len(X_train), 'val': len(X_val), 'test': len(X_test)}
print(f'Train: {SPLIT_INFO["train"]} | Val: {SPLIT_INFO["val"]} | Test: {SPLIT_INFO["test"]}')

In [ ]:
# CELDA 5: ENTRENAMIENTO
results = []
best_global = {'test_acc': 0}
start_time = time.time()

def ev(name, model):
    model.fit(X_train, y_train)
    va = accuracy_score(y_val, model.predict(X_val))
    ta = accuracy_score(y_test, model.predict(X_test))
    tp = model.predict_proba(X_test)
    tl = log_loss(y_test, tp)
    br = sum(brier_score_loss((y_test == i).astype(int), tp[:, i]) for i in range(3)) / 3
    tr = accuracy_score(y_train, model.predict(X_train))
    fi = dict(zip(avail, model.feature_importances_)) if hasattr(model, 'feature_importances_') else {}
    return {
        'model': name,
        'val_acc': round(va * 100, 2),
        'test_acc': round(ta * 100, 2),
        'logloss': round(tl, 4),
        'brier': round(br, 4),
        'train_acc': round(tr * 100, 2),
        'gap': round((tr - ta) * 100, 2),
        'feature_importance': fi
    }

T = TRIALS_PER_MODEL

models_config = [
    ('XGBoost', lambda t: xgb.XGBClassifier(
        n_estimators=t.suggest_int('n', 200, 1000),
        max_depth=t.suggest_int('d', 4, 12),
        learning_rate=t.suggest_float('lr', 0.01, 0.2, log=True),
        subsample=t.suggest_float('ss', 0.6, 1.0),
        colsample_bytree=t.suggest_float('cs', 0.5, 1.0),
        reg_alpha=t.suggest_float('a', 1e-3, 10, log=True),
        reg_lambda=t.suggest_float('l', 1e-3, 10, log=True),
        min_child_weight=t.suggest_int('mcw', 1, 10),
        gamma=t.suggest_float('g', 0, 1),
        objective='multi:softprob', eval_metric='mlogloss',
        random_state=42, verbosity=0)),
    ('LightGBM', lambda t: lgb.LGBMClassifier(
        n_estimators=t.suggest_int('n', 200, 1000),
        max_depth=t.suggest_int('d', 4, 12),
        learning_rate=t.suggest_float('lr', 0.01, 0.2, log=True),
        subsample=t.suggest_float('ss', 0.6, 1.0),
        colsample_bytree=t.suggest_float('cs', 0.5, 1.0),
        reg_alpha=t.suggest_float('a', 1e-3, 10, log=True),
        reg_lambda=t.suggest_float('l', 1e-3, 10, log=True),
        num_leaves=t.suggest_int('nl', 31, 200),
        min_child_samples=t.suggest_int('mcs', 5, 50),
        random_state=42, verbose=-1)),
    ('CatBoost', lambda t: CatBoostClassifier(
        iterations=t.suggest_int('it', 200, 800),
        depth=t.suggest_int('d', 4, 10),
        learning_rate=t.suggest_float('lr', 0.01, 0.2, log=True),
        l2_leaf_reg=t.suggest_float('l2', 1, 10),
        random_state=42, verbose=0)),
    ('RandomForest', lambda t: RandomForestClassifier(
        n_estimators=t.suggest_int('n', 200, 800),
        max_depth=t.suggest_int('d', 6, 20),
        min_samples_split=t.suggest_int('mss', 2, 20),
        min_samples_leaf=t.suggest_int('msl', 1, 10),
        max_features=t.suggest_categorical('mf', ['sqrt', 'log2', 0.5]),
        random_state=42, n_jobs=-1)),
    ('GradientBoosting', lambda t: GradientBoostingClassifier(
        n_estimators=t.suggest_int('n', 200, 800),
        max_depth=t.suggest_int('d', 4, 10),
        learning_rate=t.suggest_float('lr', 0.01, 0.2, log=True),
        subsample=t.suggest_float('ss', 0.6, 1.0),
        min_samples_split=t.suggest_int('mss', 2, 20),
        random_state=42)),
]

for name, builder in models_config:
    print(f'{name} ({T} trials)...')
    def obj(t, b=builder, n=name):
        m = b(t)
        if n == 'XGBoost':
            m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        else:
            m.fit(X_train, y_train)
        return accuracy_score(y_val, m.predict(X_val))
    s = optuna.create_study(direction='maximize')
    s.optimize(obj, n_trials=T, show_progress_bar=True)
    bp = s.best_trial.params
    m = builder(optuna.trial.FixedTrial(bp))
    r = ev(name, m)
    r['params'] = bp
    results.append(r)
    if r['test_acc'] > best_global['test_acc']:
        best_global = r
    print(f'  Val:{r["val_acc"]}% Test:{r["test_acc"]}% Gap:{r["gap"]}%')

TOTAL_TIME = round(time.time() - start_time)
print(f'\nBEST: {best_global["model"]} {best_global["test_acc"]}% ({TOTAL_TIME}s)')

In [ ]:
# CELDA 6: GENERAR REPORTE HTML
from datetime import datetime

best_fi = best_global.get('feature_importance', {})
fi_sorted = sorted(best_fi.items(), key=lambda x: -x[1])[:15]
fi_rows = ''
for feat, score in fi_sorted:
    pct = round(score * 100, 2)
    bar_w = min(pct * 8, 100)
    fi_rows += f'<tr><td>{feat}</td><td>{pct}%</td><td><div style="background:#334155;height:10px;border-radius:5px"><div style="background:#3b82f6;height:10px;border-radius:5px;width:{bar_w}%"></div></div></td></tr>\n'

model_rows = ''
for r in sorted(results, key=lambda x: -x['test_acc']):
    gc = '#f87171' if r['gap'] > 10 else '#fbbf24' if r['gap'] > 5 else '#34d399'
    model_rows += f'<tr><td><strong>{r["model"]}</strong></td><td>{r["val_acc"]}%</td><td><strong>{r["test_acc"]}%</strong></td><td>{r["logloss"]}</td><td>{r["brier"]}</td><td>{r["train_acc"]}%</td><td style="color:{gc}">{r["gap"]}%</td></tr>\n'

bp_rows = ''
for k, v in best_global.get('params', {}).items():
    bp_rows += f'<tr><td><code>{k}</code></td><td>{v}</td></tr>\n'

now = datetime.now().strftime('%Y-%m-%d %H:%M')
bn = BATCH_NUM
ba = best_global['test_acc']
bm = best_global['model']
bl = best_global['logloss']
bg = best_global['gap']
dt = DATA_INFO['total']
dl = DATA_INFO['leagues']
dfrom = DATA_INFO['from']
dto = DATA_INFO['to']
ff = FEAT_INFO['features']
fr = FEAT_INFO['rows']
st = SPLIT_INFO['train']
sv = SPLIT_INFO['val']
ste = SPLIT_INFO['test']
tt = TOTAL_TIME
trials = TRIALS_PER_MODEL * 5

html = f'''<!DOCTYPE html><html lang="es"><head><meta charset="UTF-8">
<title>Batch {bn}</title>
<style>
body{{font-family:Arial,sans-serif;background:#0f172a;color:#e2e8f0;max-width:1100px;margin:0 auto;padding:32px}}
.card{{background:#1e293b;border:1px solid #334155;border-radius:16px;padding:24px;margin-bottom:16px}}
table{{width:100%;border-collapse:collapse;font-size:13px}}
th{{text-align:left;padding:10px;color:#94a3b8;font-size:11px;text-transform:uppercase;border-bottom:1px solid #334155}}
td{{padding:10px;border-bottom:1px solid rgba(51,65,85,.4)}}
.kpi{{text-align:center;padding:20px}}
.kpi .val{{font-size:32px;font-weight:800;line-height:1}}
.kpi .lbl{{font-size:11px;color:#94a3b8;margin-top:4px}}
</style></head><body>

<div style="display:flex;align-items:center;gap:16px;margin-bottom:32px">
<div style="width:48px;height:48px;border-radius:12px;background:linear-gradient(135deg,#3b82f6,#8b5cf6);display:flex;align-items:center;justify-content:center;color:white;font-size:20px;font-weight:800">{bn}</div>
<div><h1 style="font-size:24px;font-weight:800;margin:0">Batch {bn}</h1>
<p style="font-size:13px;color:#94a3b8;margin:0">{now} | {tt}s | {trials} trials</p></div></div>

<div style="display:grid;grid-template-columns:repeat(5,1fr);gap:12px;margin-bottom:24px">
<div class="card kpi"><div class="val" style="color:#60a5fa">{ba}%</div><div class="lbl">Accuracy</div></div>
<div class="card kpi"><div class="val" style="color:#34d399;font-size:18px">{bm}</div><div class="lbl">Modelo</div></div>
<div class="card kpi"><div class="val" style="color:#a78bfa">{bl}</div><div class="lbl">Log Loss</div></div>
<div class="card kpi"><div class="val" style="color:#fbbf24">{bg}%</div><div class="lbl">Gap</div></div>
<div class="card kpi"><div class="val" style="color:#22d3ee">{dt}</div><div class="lbl">Partidos</div></div></div>

<div class="card"><h2 style="font-weight:700;margin-bottom:16px">Config</h2>
<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:16px;font-size:13px">
<div><p style="color:#94a3b8">Dataset</p><p><strong>{dt}</strong> partidos, <strong>{dl}</strong> ligas ({dfrom} &rarr; {dto})</p></div>
<div><p style="color:#94a3b8">Features</p><p><strong>{ff}</strong> features en <strong>{fr}</strong> rows</p></div>
<div><p style="color:#94a3b8">Split</p><p>Train:<strong>{st}</strong> Val:<strong>{sv}</strong> Test:<strong>{ste}</strong></p></div></div></div>

<div class="card"><h2 style="font-weight:700;margin-bottom:16px">Resultados</h2>
<table><tr><th>Modelo</th><th>Val</th><th>Test</th><th>LogLoss</th><th>Brier</th><th>Train</th><th>Gap</th></tr>
{model_rows}</table></div>

<div class="card"><h2 style="font-weight:700;margin-bottom:16px">Mejor: {bm}</h2>
<table><tr><th>Param</th><th>Valor</th></tr>{bp_rows}</table></div>

<div class="card"><h2 style="font-weight:700;margin-bottom:16px">Feature Importance</h2>
<table><tr><th>Feature</th><th>Imp</th><th></th></tr>{fi_rows}</table></div>

<p style="text-align:center;font-size:11px;color:#475569">Batch {bn} | {now} | Mondial-Xboost</p></body></html>'''

DP = '/content/drive/MyDrive/Mondial-Xboost/data/'
os.makedirs(DP, exist_ok=True)
with open(DP + f'batch_{bn}.html', 'w', encoding='utf-8') as f:
    f.write(html)
with open(DP + f'batch_{bn}_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f'\nHTML: {DP}batch_{bn}.html')
print(f'JSON: {DP}batch_{bn}_results.json')
print(f'\n>>> Abrir batch_{bn}.html en el navegador <<<')